In [0]:
%sql
CREATE OR REPLACE TABLE anvisa.g_vigimed_features_ml AS

WITH 
-- BLOCO 1: Padronização e agrupamento de todas as variações de VIA_ADMINISTRACAO.
med_vias_padronizadas AS (
  SELECT
    IDENTIFICACAO_NOTIFICACAO,
    PRINCIPIOS_ATIVOS_WHODRUG,
    NOME_MEDICAMENTO_WHODRUG,
    RELACAO_MEDICAMENTO_EVENTO,
    VIA_ADMINISTRACAO,
    CASE
      -- 1. Intravenosa / Endovenosa / Intravascular
      WHEN VIA_ADMINISTRACAO RLIKE '(INTRA.*VEN|ENDO.*VEN|VENOS|VENOO|VEDNOS|ENOSA|NA VEIA|INTRA.*VASC|INTRV|INRAV|ITRAV|INTAV|INTRAVE|INTRAVN|ENODV|ENDON|ENDOV|ENDEV|ENDIV|ENFOV|ENVOD|RNDOV|INDOV|EDOV|ANDOV|ENCOV|ESNDOV|ENDEN|ENDOE|ENDOD|ENOV|INTREV|INTRTAV|INTARV|INTRRA|INYTRA|IMTRAV|\\bIV\\b|\\bEV\\b|\\bE\\s+V\\b|\\|V|\\bI\\.V|\\bE\\.V|\\bIVD\\b|\\bEVB\\b|\\bEVI\\b|\\bEVC\\b|\\bEVE\\b|\\bIVI\\b|\\bIVE\\b|\\bIVAD\\b|\\bIVBC\\b|\\bIBVIC\\b|\\bIVBIC\\b|\\bBIC\\b|INFUS|BOLUS|BOULOS|PORT.*CATH|PERMICATH|PICC|ACESSO.*CENTRAL|CATETER.*CENTRAL|ACESSO.*PERIF|CATETER.*PERIF|SUBCLAVIA|FEMORAL|\\bCVC\\b|EXTRACORPOR|ECMO|HEMODIALISE|DIALISE|ARTERIAL|INTRACARD|INTRACORON|SORO FISIOLOGICO|\\bSF\\b|\\bSG\\b)' 
        THEN 'INTRAVENOSA_INTRAVASCULAR'

      -- 2. Intramuscular
      WHEN VIA_ADMINISTRACAO RLIKE '(MUSCUL|INTRAMASC|INTRAMUS|DELTOID|DEITOID|VASTO|VL\\.|\\bVLE\\b|GLUTE|\\bIM\\b|\\bI\\.M|\\bI M\\b|BUMBUM|COXA)' 
        THEN 'INTRAMUSCULAR'

      -- 3. Subcutânea / Intradérmica / Hipodermóclise
      WHEN VIA_ADMINISTRACAO RLIKE '(SUB.*CUT|SUB.*CULT|SUNCUT|SUBCT|SUBCU|SUCUT|SUCBUT|SUSCUT|SUBSTUT|SUBUT|HIPODERMOCLISE|\\bHDC\\b|\\bSC\\b|\\bS\\.C|SUB.*DERM|INTRA.*DERM|CANETA|AUTOINJETOR)' 
        THEN 'SUBCUTANEA_INTRADERMICA'

      -- 4. Enteral por Sonda / Ostomias
      WHEN VIA_ADMINISTRACAO RLIKE '(SONDA|SONA|GASTRO|GTM|JEJUN|NASO.*ENT|NASO.*GAST|ORO.*GAST|ORO.*ENT|NASO.*DUOD|\\bSNE\\b|\\bSNG\\b|\\bSOG\\b|\\bGTT\\b|\\bVSNE\\b|TUBO)' 
        THEN 'ENTERAL_SONDA_OSTOMIA'

      -- 5. Respiratória / Inalatória / Nasal
      WHEN VIA_ADMINISTRACAO RLIKE '(INALA|INHAL|INAL\\b|RESPIRAT|NEBULIZ|AEROSOL|AEROSSOL|PULMON|ALVEOL|ENDOTRAQ|INTRATRAQ|BRONQU|NASAL|NARIZ|LARING|AREA SUPERIOR)' 
        THEN 'RESPIRATORIA_INALATORIA_NASAL'

      -- 6. Neuroaxial (Intratecal/Peridural/Raqui) e Intracavitária
      WHEN VIA_ADMINISTRACAO RLIKE '(INTRATEC|INTRATHEC|INTRETEC|INTRATERC|INTRATECT|INTRETEG|INTERTEC|INTARTEC|INTRECAL|INTRATRC|ENTRATEC|INTERTERC|INTRA.*TEC|PERIDU|EPIDU|RAQUI|SUBARAC|SUBCARAC|SUBRADN|SUBDURAL|ESPINHAL|INTRAMEDUL|CEREBROVENT|PUNCAO LOMBAR|PERINEURAL|LOCORREGIONAL|ANESTESIA REGIONAL|SUBCLAVICULAR|PERITON|PERICARD|HEPATIC|PLEURAL|SINOVIAL|ARTICUL|VESICAL|INTRAVES|INTREVES|CAVIT|CAVERNOS|LESIONAL|INFILTR|\\bIT\\b|\\bIP\\b)' 
        THEN 'NEUROAXIAL_INTRACAVITARIA'

      -- 7. Oftálmica / Ocular / Otológica
      WHEN VIA_ADMINISTRACAO RLIKE '(OFTALM|OFITALM|OFTAMOL|OPHTHALM|OCULAR|OLHO|EYE|COLIRIO|VITRE|RETIN|CONJUNTIV|SUBCONJUN|OTOLOG|AURICUL|OUVIDO)' 
        THEN 'OFTALMICA_OTOLOGICA'

      -- 8. Tópica / Transdérmica / Cutânea
      WHEN VIA_ADMINISTRACAO RLIKE '(TOPIC|TRANSDERM|TRASDERM|TRANDERM|TRANSCUT|PERCUT|CUTANE|CULTANE|DERMAT|DERMIC|DERME|EPILESION|POMADA|GEL\\b|ADESIV|ADES\\b|COURO CABELUDO|CAPILAR|PESCOCO|\\bTD\\b|\\bTO\\b)' 
        THEN 'TOPICA_TRANSDERMICA_CUTANEA'

      -- 9. Retal / Ginecológica / Urogenital
      WHEN VIA_ADMINISTRACAO RLIKE '(RETAL|RECTAL|RETA\\b|SUPOSIT|ENEMA|VAGIN|\\bVG\\b|UTERIN|ENDOCERVIC|URETRAL|URETHRAL|\\bVR\\b|\\bV\\.R)' 
        THEN 'RETAL_GINECOLOGICA_UROGENITAL'

      -- 10. Oral / Bucal / Sublingual
      WHEN VIA_ADMINISTRACAO RLIKE '(\\bORAL\\b|VIAORAL|ORLA|OREAL|0RAL|ORAC|\\bORA\\b|\\bOAL\\b|\\bVO\\b|\\bV\\.O|\\bV/O\\b|\\bV\\.0\\b|\\bPO\\b|SUB.*LING|SULING|\\bSL\\b|BUCAL|BUCCAL|SUBMUCOS|OROMUC|OROFARING|GARGAREJO|COMPRIMIDO|\\bCP\\b|CAPSUL|TABLET|XAROPE|SACHE|GOTAS|COLHERES|BOCA|BOCHECHA|MASTIG|CAFE|JANTAR|ALMOCO|REFEIC|ENTERAL|INTESTIN|INTRALUM)' 
        THEN 'ORAL_BUCAL_SUBLINGUAL'

      -- 11. Parenteral Inespecífica
      WHEN VIA_ADMINISTRACAO RLIKE '(PARENTERAL|INJETAV|INJEC|INJAC|INJECTION|INEJT|INFETAV|INETAV|SOL INJ|AMPOLA|MAPOLA|\\bAMP\\b|FRASCO|VIALS|LIOFIL|SERINGA|VACINA|BRACO|MEMBRO|ANTEBRACO|BARRIGA|ABDOMEN|PULSOS|\\bMSD\\b|\\bMSE\\b)' 
        THEN 'PARENTERAL_INESPECIFICA'

      -- 12. Exposição Materno-Fetal
      WHEN VIA_ADMINISTRACAO RLIKE '(PLACENT|MATERNO|TRANSMAM|GRAVIDEZ|MAE|PAI|FATHER|PARTNER|PROGENITOR)' 
        THEN 'EXPOSICAO_MATERNO_FETAL'

      -- 13. Não Informada, Desconhecida ou Inconsistente
      ELSE 'NAO_INFORMADA_OU_INCONSISTENTE'
    END AS GRUPO_VIA_ADMINISTRACAO
  FROM anvisa.s_vigimed_medicamentos
),

-- BLOCO 2: Agregação de Medicamentos e Cruzamento Regulatório (1:N -> 1:1)
agg_medicamentos AS (
  SELECT
    med.IDENTIFICACAO_NOTIFICACAO,
    
    -- Métricas de Polifarmácia
    count(DISTINCT med.PRINCIPIOS_ATIVOS_WHODRUG) AS QTD_PRINCIPIOS_ATIVOS,
    count(DISTINCT med.NOME_MEDICAMENTO_WHODRUG)   AS QTD_MEDICAMENTOS_TOTAL,
    
    -- Contagem por Papel Causal do Medicamento
    count(DISTINCT CASE WHEN med.RELACAO_MEDICAMENTO_EVENTO LIKE '%SUSPEITO%'     THEN med.PRINCIPIOS_ATIVOS_WHODRUG END) AS QTD_FARMACOS_SUSPEITOS,
    count(DISTINCT CASE WHEN med.RELACAO_MEDICAMENTO_EVENTO LIKE '%CONCOMITANTE%' THEN med.PRINCIPIOS_ATIVOS_WHODRUG END) AS QTD_FARMACOS_CONCOMITANTES,
    count(DISTINCT CASE WHEN med.RELACAO_MEDICAMENTO_EVENTO LIKE '%INTERACAO%'    THEN med.PRINCIPIOS_ATIVOS_WHODRUG END) AS QTD_FARMACOS_INTERACAO,

    -- Flags Binárias (0/1) das 13 Categorias Consolidadas de Via de Administração
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'INTRAVENOSA_INTRAVASCULAR'      THEN 1 ELSE 0 END) AS FLG_VIA_INTRAVENOSA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'INTRAMUSCULAR'                  THEN 1 ELSE 0 END) AS FLG_VIA_INTRAMUSCULAR,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'SUBCUTANEA_INTRADERMICA'        THEN 1 ELSE 0 END) AS FLG_VIA_SUBCUTANEA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'ENTERAL_SONDA_OSTOMIA'          THEN 1 ELSE 0 END) AS FLG_VIA_ENTERAL_SONDA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'RESPIRATORIA_INALATORIA_NASAL'  THEN 1 ELSE 0 END) AS FLG_VIA_RESPIRATORIA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'NEUROAXIAL_INTRACAVITARIA'      THEN 1 ELSE 0 END) AS FLG_VIA_NEUROAXIAL_CAVITARIA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'OFTALMICA_OTOLOGICA'            THEN 1 ELSE 0 END) AS FLG_VIA_OFTALMICA_OTOLOGICA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'TOPICA_TRANSDERMICA_CUTANEA'    THEN 1 ELSE 0 END) AS FLG_VIA_TOPICA_CUTANEA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'RETAL_GINECOLOGICA_UROGENITAL'  THEN 1 ELSE 0 END) AS FLG_VIA_RETAL_UROGENITAL,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'ORAL_BUCAL_SUBLINGUAL'          THEN 1 ELSE 0 END) AS FLG_VIA_ORAL,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'PARENTERAL_INESPECIFICA'        THEN 1 ELSE 0 END) AS FLG_VIA_PARENTERAL_INESPECIFICA,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'EXPOSICAO_MATERNO_FETAL'        THEN 1 ELSE 0 END) AS FLG_VIA_MATERNO_FETAL,
    max(CASE WHEN med.GRUPO_VIA_ADMINISTRACAO = 'NAO_INFORMADA_OU_INCONSISTENTE' THEN 1 ELSE 0 END) AS FLG_VIA_NAO_INFORMADA,

    -- Flags de Categoria Regulatória (Enriquecimento com DADOS_ABERTOS_MEDICAMENTOS)
    max(CASE WHEN reg.CATEGORIA_REGULATORIA LIKE '%GENERICO%'  THEN 1 ELSE 0 END) AS FLG_USA_GENERICO,
    max(CASE WHEN reg.CATEGORIA_REGULATORIA LIKE '%SIMILAR%'   THEN 1 ELSE 0 END) AS FLG_USA_SIMILAR,
    max(CASE WHEN reg.CATEGORIA_REGULATORIA LIKE '%NOVO%'      THEN 1 ELSE 0 END) AS FLG_USA_NOVO,
    max(CASE WHEN reg.CATEGORIA_REGULATORIA LIKE '%BIOLOGICO%' THEN 1 ELSE 0 END) AS FLG_USA_BIOLOGICO

  FROM med_vias_padronizadas med
  LEFT JOIN anvisa.s_dados_abertos_medicamentos reg
    ON med.PRINCIPIOS_ATIVOS_WHODRUG = reg.PRINCIPIO_ATIVO
  GROUP BY med.IDENTIFICACAO_NOTIFICACAO
),

-- BLOCO 3: Agregação de Reações Adversas e Construção da Variável Alvo (1:N -> 1:1)
agg_reacoes AS (
  SELECT
    reac.IDENTIFICACAO_NOTIFICACAO,
    
    -- Volume e complexidade sintomática
    count(DISTINCT reac.PT)  AS QTD_REACOES_DISTINTAS,
    count(DISTINCT reac.SOC) AS QTD_SISTEMAS_AFETADOS,

    -- Flags por Sistema Orgânico Afetado (MedDRA SOC)
    max(CASE WHEN reac.SOC LIKE '%CARDIAC%'         THEN 1 ELSE 0 END) AS FLG_SOC_CARDIACO,
    max(CASE WHEN reac.SOC LIKE '%GASTROINTESTINAL%' THEN 1 ELSE 0 END) AS FLG_SOC_GASTROINTESTINAL,
    max(CASE WHEN reac.SOC LIKE '%RESPIRATOR%'       THEN 1 ELSE 0 END) AS FLG_SOC_RESPIRATORIO,
    max(CASE WHEN reac.SOC LIKE '%SISTEMA NERVOSO%'  THEN 1 ELSE 0 END) AS FLG_SOC_NERVOSO,
    max(CASE WHEN reac.SOC LIKE '%PELE%' OR reac.SOC LIKE '%CUTANEO%' THEN 1 ELSE 0 END) AS FLG_SOC_CUTANEO,

    -- Regra de severidade para a variável Target de Machine Learning
    max(
      CASE 
        WHEN reac.FLG_GRAVE = 1 THEN 1
        WHEN reac.GRAVIDADE IN ('HOSPITALIZACAO', 'AMEACA A VIDA', 'INCAPACIDADE', 'ANOMALIA CONGENITA') THEN 1
        WHEN reac.DESFECHO IN ('OBITO', 'RECUPERACAO COM SEQUELA', 'NAO RECUPERADO') THEN 1
        ELSE 0 
      END
    ) AS TARGET_IS_GRAVE

  FROM anvisa.s_vigimed_reacoes reac
  GROUP BY reac.IDENTIFICACAO_NOTIFICACAO
)

-- BLOCO 4: Consolidação Final na Granularidade de Notificação (1 linha por Caso)
SELECT
  -- Identificador Único
  notif.IDENTIFICACAO_NOTIFICACAO,

  -- Variáveis Demográficas e Antropométricas
  notif.IDADE_MOMENTO_REACAO AS IDADE_ANOS,
  notif.SEXO,
  notif.UF,
  notif.PESO_KG,
  notif.ALTURA_CM,
  round(
    CASE 
      WHEN notif.PESO_KG > 0 AND notif.ALTURA_CM > 0 
      THEN notif.PESO_KG / power(notif.ALTURA_CM / 100.0, 2)
      ELSE NULL 
    END, 2
  ) AS IMC,

  -- Condições Maternas e Metadados do Relato
  coalesce(notif.FLG_GESTANTE, 0) AS FLG_GESTANTE,
  coalesce(notif.FLG_LACTANTE, 0) AS FLG_LACTANTE,
  notif.TIPO_NOTIFICACAO,
  notif.RECEBIDO_DE,
  datediff(notif.DATA_INCLUSAO_SISTEMA, notif.DATA_NOTIFICACAO) AS DIAS_ATE_INCLUSAO_SISTEMA,

  -- Métricas de Polifarmácia e Causalidade
  coalesce(med.QTD_PRINCIPIOS_ATIVOS, 0)      AS QTD_PRINCIPIOS_ATIVOS,
  coalesce(med.QTD_MEDICAMENTOS_TOTAL, 0)     AS QTD_MEDICAMENTOS_TOTAL,
  coalesce(med.QTD_FARMACOS_SUSPEITOS, 0)     AS QTD_FARMACOS_SUSPEITOS,
  coalesce(med.QTD_FARMACOS_CONCOMITANTES, 0) AS QTD_FARMACOS_CONCOMITANTES,
  coalesce(med.QTD_FARMACOS_INTERACAO, 0)     AS QTD_FARMACOS_INTERACAO,
  
  -- Todas as 13 Categorias de Via de Administração Mapeadas
  coalesce(med.FLG_VIA_INTRAVENOSA, 0)             AS FLG_VIA_INTRAVENOSA,
  coalesce(med.FLG_VIA_INTRAMUSCULAR, 0)           AS FLG_VIA_INTRAMUSCULAR,
  coalesce(med.FLG_VIA_SUBCUTANEA, 0)              AS FLG_VIA_SUBCUTANEA,
  coalesce(med.FLG_VIA_ENTERAL_SONDA, 0)           AS FLG_VIA_ENTERAL_SONDA,
  coalesce(med.FLG_VIA_RESPIRATORIA, 0)            AS FLG_VIA_RESPIRATORIA,
  coalesce(med.FLG_VIA_NEUROAXIAL_CAVITARIA, 0)    AS FLG_VIA_NEUROAXIAL_CAVITARIA,
  coalesce(med.FLG_VIA_OFTALMICA_OTOLOGICA, 0)     AS FLG_VIA_OFTALMICA_OTOLOGICA,
  coalesce(med.FLG_VIA_TOPICA_CUTANEA, 0)          AS FLG_VIA_TOPICA_CUTANEA,
  coalesce(med.FLG_VIA_RETAL_UROGENITAL, 0)        AS FLG_VIA_RETAL_UROGENITAL,
  coalesce(med.FLG_VIA_ORAL, 0)                    AS FLG_VIA_ORAL,
  coalesce(med.FLG_VIA_PARENTERAL_INESPECIFICA, 0) AS FLG_VIA_PARENTERAL_INESPECIFICA,
  coalesce(med.FLG_VIA_MATERNO_FETAL, 0)           AS FLG_VIA_MATERNO_FETAL,
  coalesce(med.FLG_VIA_NAO_INFORMADA, 0)           AS FLG_VIA_NAO_INFORMADA,

  -- Categorias Regulatórias dos Medicamentos Envolvidos
  coalesce(med.FLG_USA_GENERICO, 0)  AS FLG_USA_GENERICO,
  coalesce(med.FLG_USA_SIMILAR, 0)   AS FLG_USA_SIMILAR,
  coalesce(med.FLG_USA_NOVO, 0)      AS FLG_USA_NOVO,
  coalesce(med.FLG_USA_BIOLOGICO, 0) AS FLG_USA_BIOLOGICO,

  -- Perfil de Sintomas e Sistemas Afetados
  coalesce(reac.QTD_REACOES_DISTINTAS, 0)    AS QTD_REACOES_DISTINTAS,
  coalesce(reac.QTD_SISTEMAS_AFETADOS, 0)    AS QTD_SISTEMAS_AFETADOS,
  coalesce(reac.FLG_SOC_CARDIACO, 0)         AS FLG_SOC_CARDIACO,
  coalesce(reac.FLG_SOC_GASTROINTESTINAL, 0) AS FLG_SOC_GASTROINTESTINAL,
  coalesce(reac.FLG_SOC_RESPIRATORIO, 0)     AS FLG_SOC_RESPIRATORIO,
  coalesce(reac.FLG_SOC_NERVOSO, 0)          AS FLG_SOC_NERVOSO,
  coalesce(reac.FLG_SOC_CUTANEO, 0)          AS FLG_SOC_CUTANEO,

  -- Variável Dependente (Target)
  coalesce(reac.TARGET_IS_GRAVE, notif.FLG_GRAVE, 0) AS TARGET_IS_GRAVE

FROM anvisa.s_vigimed_notificacoes notif
LEFT JOIN agg_medicamentos med
  ON notif.IDENTIFICACAO_NOTIFICACAO = med.IDENTIFICACAO_NOTIFICACAO
LEFT JOIN agg_reacoes reac
  ON notif.IDENTIFICACAO_NOTIFICACAO = reac.IDENTIFICACAO_NOTIFICACAO;